# LSTM vs HAR: the 2 x 2 grid (cutoff x memory length)

What this notebook does, in one run:

1. the **same LSTM** as `03_lstm.ipynb`, for both cutoffs (2019, 2021) and both memory lengths (22 and 66 days);
2. **retraining on all training years**: after early stopping finds the best number of epochs, the network is trained again on fit + validation years for that many epochs, so it sees the same years as HAR;
3. **the same sigma2 rule for every model**: the error variance on the validation year (the protocol rule). In the last notebook HAR used its training errors;
4. a **Diebold-Mariano test** of each model against HAR, on the daily average QLIKE.

Set `QUICK = True` in the first cell for a one-minute test on 3 stocks (the numbers mean nothing then).

In [1]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.linear_model import LinearRegression

QUICK = False                                   # True = 3 stocks, 1 seed, 3 epochs: only to check that it runs
SEEDS = (0,) if QUICK else (0, 1, 2)
MAX_EPOCHS = 3 if QUICK else 30

# project root = the folder that contains code/
ROOT = Path.cwd() if (Path.cwd() / "code").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "code"))          # so we can import Jean's code
from econometrics.config import RunConfig
from econometrics.data import build_panel
from econometrics.evaluation import dm_test, qlike      # needs: pip install arch

## 1. Load the cleaned data (Jean's `build_panel`)

In [2]:
panel, mask_log = build_panel(RunConfig(dataset="stocks", quick=QUICK))
panel = panel.reset_index(drop=True)

FEATURES = ["x_day", "x_week", "x_month", "m_day", "m_week", "jump_share"]   # HAR + market + jump
CHANNELS = ["y", "m_day", "jump_share"]                                      # what the LSTM reads, day by day
panel["eligible"] = panel[FEATURES + ["target", "target_rv"]].notna().all(axis=1)

el = panel[panel["eligible"]].reset_index(drop=True)       # the rows every model uses
y_all = el["target"].to_numpy()                            # tomorrow's log variance
print(f"{len(panel):,} stock-days, {len(el):,} usable, {len(mask_log)} masked days")
for cutoff in ["2019-01-01", "2021-01-01"]:
    print(cutoff, "test rows:", f"{(el['date'] >= cutoff).sum():,}", "(protocol: 78,605 from 2019, 58,405 from 2021)")

119,000 stock-days, 117,982 usable, 6 masked days
2019-01-01 test rows: 78,605 (protocol: 78,605 from 2019, 58,405 from 2021)
2021-01-01 test rows: 58,405 (protocol: 78,605 from 2019, 58,405 from 2021)


## 2. Building blocks

- `make_windows(seq)`: for each row, the last `seq` days of the same stock (oldest first, ending today). A masked day takes the previous day's value, and the first rows of a stock repeat their first day, so every row has a full window. The test rows always have real history. Three `assert` lines check that the windows line up with the HAR features.
- `LSTM`: reads the days in order, keeps a memory of 32 numbers, one linear layer turns the last memory into the forecast.
- `train_early_stop`: trains on the fit years and keeps the epoch with the best validation loss.
- `train_fixed`: trains on all training years for a given number of epochs (the retraining).

In [3]:
WINDOWS = {}

def make_windows(seq):
    if seq in WINDOWS:
        return WINDOWS[seq]
    windows = np.full((len(panel), seq, len(CHANNELS)), np.nan, dtype=np.float32)
    for symbol, g in panel.groupby("symbol"):                       # stocks never mix
        values = g[CHANNELS].ffill().bfill().to_numpy(dtype=np.float32)
        padded = np.concatenate([np.repeat(values[:1], seq - 1, axis=0), values])
        windows[g.index.to_numpy()] = sliding_window_view(padded, (seq, len(CHANNELS)))[:, 0]
    X = windows[panel["eligible"].to_numpy()]
    assert not np.isnan(X).any()
    assert np.allclose(X[:, -1, 0], el["x_day"], atol=1e-4)
    assert np.allclose(X[:, -5:, 0].mean(1), el["x_week"], atol=1e-4)
    assert np.allclose(X[:, -22:, 0].mean(1), el["x_month"], atol=1e-4)
    WINDOWS[seq] = X
    return X


class LSTM(nn.Module):
    def __init__(self, n_channels, hidden=32):
        super().__init__()
        self.lstm = nn.LSTM(n_channels, hidden, batch_first=True)
        self.out = nn.Linear(hidden, 1)

    def forward(self, x):
        states, _ = self.lstm(x)                      # memory after each day
        return self.out(states[:, -1]).squeeze(-1)    # forecast from the memory after the last day


to_t = lambda a: torch.tensor(a, dtype=torch.float32)


def predict(model, X, batch=8192):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(X[i:i + batch]) for i in range(0, len(X), batch)]).numpy()


def new_model(seed, n_channels, hidden=32, lr=1e-3):
    torch.manual_seed(seed)
    model = LSTM(n_channels, hidden)
    return model, torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)


def run_epoch(model, optimizer, X, y, batch=512):
    model.train()
    order = torch.randperm(len(X))                    # new random order of the rows each epoch
    for i in range(0, len(X), batch):
        rows = order[i:i + batch]
        loss = nn.functional.mse_loss(model(X[rows]), y[rows])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


def train_early_stop(X_fit, y_fit, X_val, y_val, seed, max_epochs=MAX_EPOCHS, patience=5):
    model, optimizer = new_model(seed, X_fit.shape[2])
    best_val, best_state, best_epoch, wait = float("inf"), None, 0, 0
    for epoch in range(1, max_epochs + 1):
        run_epoch(model, optimizer, X_fit, y_fit)
        val_loss = nn.functional.mse_loss(torch.from_numpy(predict(model, X_val)), y_val).item()
        if val_loss < best_val - 1e-6:
            best_val, best_epoch, wait = val_loss, epoch, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)                 # back to the best epoch
    return model, best_epoch


def train_fixed(X_train, y_train, seed, epochs):
    model, optimizer = new_model(seed, X_train.shape[2])
    for _ in range(epochs):
        run_epoch(model, optimizer, X_train, y_train)
    return model

## 3. One experiment = one cutoff and one memory length

For each seed: train with early stopping (fit years, validation year), then retrain on all years before the cutoff for the same number of epochs. Forecasts are the average of the seeds.

- **sigma2** (the `exp(forecast + sigma2 / 2)` correction) = variance of the validation errors, for HAR and for the LSTM alike. For the retrained LSTM it comes from the early-stopped networks, because the retrained ones have already seen the validation year.
- Final HAR = regression on all training years (as before).

In [4]:
def run_config(cutoff, seq, seeds=SEEDS):
    t0 = time.time()
    cutoff = pd.Timestamp(cutoff)
    val_start = cutoff - pd.DateOffset(years=1)
    d = el["date"]
    train = (d < cutoff).to_numpy()                   # fit + validation years
    fit = (d < val_start).to_numpy()
    val = ((d >= val_start) & (d < cutoff)).to_numpy()
    test = (d >= cutoff).to_numpy()
    y = y_all
    rv = el.loc[test, "target_rv"].to_numpy()
    days = el.loc[test, "date"].to_numpy()

    # ---- baselines
    rw_test = el.loc[test, "x_day"].to_numpy()
    har = LinearRegression().fit(el.loc[train, FEATURES], y[train])           # final HAR: all training years
    har_test = har.predict(el.loc[test, FEATURES])
    har_fit = LinearRegression().fit(el.loc[fit, FEATURES], y[fit])           # only to measure validation errors
    sigma2_har = np.var(y[val] - har_fit.predict(el.loc[val, FEATURES]))

    # ---- LSTM: standardize with the fit years only
    X_seq = make_windows(seq)
    flat = X_seq[fit].reshape(-1, len(CHANNELS))
    Xn = (X_seq - flat.mean(0)) / flat.std(0)
    y_mu, y_sd = y[fit].mean(), y[fit].std()
    yn = (y - y_mu) / y_sd
    X_fit, y_fit = to_t(Xn[fit]), to_t(yn[fit])
    X_val, y_val = to_t(Xn[val]), to_t(yn[val])
    X_train, y_train = to_t(Xn[train]), to_t(yn[train])
    X_test = to_t(Xn[test])

    val_preds, early_preds, final_preds = [], [], []
    for seed in seeds:
        model, best_epoch = train_early_stop(X_fit, y_fit, X_val, y_val, seed)
        val_preds.append(predict(model, X_val))
        early_preds.append(predict(model, X_test))
        retrained = train_fixed(X_train, y_train, seed, best_epoch)
        final_preds.append(predict(retrained, X_test))
        print(f"  seed {seed}: best epoch {best_epoch}")
    unscale = lambda preds: np.mean(preds, axis=0) * y_sd + y_mu             # average the seeds, back to log RV
    lstm_val, lstm_early, lstm_final = unscale(val_preds), unscale(early_preds), unscale(final_preds)
    sigma2_lstm = np.var(y[val] - lstm_val)

    # ---- score on the same test rows
    forecasts = {                                      # (log forecast, variance forecast)
        "tomorrow = today": (rw_test, np.exp(rw_test)),
        "HAR": (har_test, np.exp(har_test + sigma2_har / 2)),
        "LSTM (early stop)": (lstm_early, np.exp(lstm_early + sigma2_lstm / 2)),
        "LSTM (retrained)": (lstm_final, np.exp(lstm_final + sigma2_lstm / 2)),
    }
    sigma2_used = {"HAR": sigma2_har, "LSTM (early stop)": sigma2_lstm, "LSTM (retrained)": sigma2_lstm}
    # the DM test works on one loss per day: the average over the stocks of that day
    daily = pd.DataFrame({name: pd.Series(qlike(rv, h)).groupby(days).mean() for name, (_, h) in forecasts.items()})

    rows = []
    for name, (log_pred, h) in forecasts.items():
        err = y[test] - log_pred
        stat, p = dm_test(daily[name], daily["HAR"]) if name != "HAR" else (np.nan, np.nan)
        rows.append({"cutoff": str(cutoff.date()), "seq": seq, "model": name, "n_test": int(test.sum()),
                     "MSE log RV": np.mean(err ** 2), "QLIKE": qlike(rv, h).mean(),
                     "sigma2 used": sigma2_used.get(name, np.nan), "test error var": np.var(err),
                     "test bias": err.mean(), "DM stat vs HAR": stat, "DM p-value": p})
    print(f"{cutoff.date()} | {seq} days | {time.time() - t0:.0f}s")
    return rows

## 4. Run the 2 x 2 grid

Four experiments. Each one prints the best epoch of every seed and its running time.

In [5]:
all_rows = []
for cutoff in ["2019-01-01", "2021-01-01"]:
    for seq in [22, 66]:
        all_rows += run_config(cutoff, seq)
results = pd.DataFrame(all_rows)

  seed 0: best epoch 13
  seed 1: best epoch 7
  seed 2: best epoch 7
2019-01-01 | 22 days | 35s
  seed 0: best epoch 14
  seed 1: best epoch 7
  seed 2: best epoch 10
2019-01-01 | 66 days | 124s
  seed 0: best epoch 6
  seed 1: best epoch 3
  seed 2: best epoch 6
2021-01-01 | 22 days | 35s
  seed 0: best epoch 6
  seed 1: best epoch 4
  seed 2: best epoch 5
2021-01-01 | 66 days | 112s


## 5. Results

Lower is better for MSE and QLIKE. The `vs HAR (%)` columns are negative when the model beats HAR. In the DM test, a negative statistic with p < 0.05 means the model is significantly better than HAR on QLIKE.

`sigma2 used` against `test error var`, and `test bias` (positive = the model under-predicts) show whether the `exp(forecast + sigma2 / 2)` correction fits the test period.

In [6]:
ref = results[results["model"] == "HAR"][["cutoff", "seq", "MSE log RV", "QLIKE"]]
ref = ref.rename(columns={"MSE log RV": "mse_har", "QLIKE": "qlike_har"})
results = results.drop(columns=[c for c in ["mse_har", "qlike_har"] if c in results]).merge(ref, on=["cutoff", "seq"])
results["MSE vs HAR (%)"] = 100 * (results["MSE log RV"] / results["mse_har"] - 1)
results["QLIKE vs HAR (%)"] = 100 * (results["QLIKE"] / results["qlike_har"] - 1)

show = ["cutoff", "seq", "model", "MSE log RV", "MSE vs HAR (%)", "QLIKE", "QLIKE vs HAR (%)", "DM p-value",
        "sigma2 used", "test error var", "test bias"]
results[show].sort_values(["cutoff", "seq", "QLIKE"]).round(4)

,cutoff,seq,model,MSE log RV,MSE vs HAR (%),QLIKE,QLIKE vs HAR (%),DM p-value,sigma2 used,test error var,test bias
1,2019-01-01,22,HAR,0.2831,0.0000,0.1820,0.0000,NaN,0.2766,0.2831,0.0021
3,2019-01-01,22,LSTM (retrained),0.2868,1.2796,0.1859,2.1656,0.3695,0.2686,0.2863,0.0221
2,2019-01-01,22,LSTM (early stop),0.2864,1.1391,0.1870,2.7618,0.2657,0.2686,0.2846,0.0421
0,2019-01-01,22,tomorrow = today,0.3860,36.3340,0.2486,36.6194,0.0000,NaN,0.3860,-0.0002
5,2019-01-01,66,HAR,0.2831,0.0000,0.1820,0.0000,NaN,0.2766,0.2831,0.0021
6,2019-01-01,66,LSTM (early stop),0.2834,0.0935,0.1843,1.2619,0.5464,0.2685,0.2828,0.0249
7,2019-01-01,66,LSTM (retrained),0.2874,1.4948,0.1875,3.0473,0.2787,0.2685,0.2851,0.0474
4,2019-01-01,66,tomorrow = today,0.3860,36.3340,0.2486,36.6194,0.0000,NaN,0.3860,-0.0002
10,2021-01-01,22,LSTM (early stop),0.2698,-1.3472,0.1748,-0.9817,0.0018,0.3552,0.2696,0.0156
11,2021-01-01,22,LSTM (retrained),0.2713,-0.7897,0.1753,-0.7108,0.0005,0.3552,0.2704,0.0307


In [7]:
out = ROOT / "results" / "dl"
out.mkdir(parents=True, exist_ok=True)
results.to_csv(out / ("lstm_grid_quick.csv" if QUICK else "lstm_grid.csv"), index=False)
print("saved to", (out / ("lstm_grid_quick.csv" if QUICK else "lstm_grid.csv")).relative_to(ROOT))

saved to results/dl/lstm_grid.csv


## How to read it

- Does the **retrained** LSTM do better than the **early stop** one? That shows what seeing all the years is worth.
- Does 66 days beat 22 days, at both cutoffs?
- Is the DM p-value below 0.05 for the LSTM against HAR? If not, call it a tie, whatever the sign of the percentage.
- Is the bias of the LSTM close to HAR's? A large bias means QLIKE is penalizing a miscalibrated level, not a worse model.